In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler
import mlflow
import warnings
warnings.filterwarnings('ignore')

# Load data
customers_df = pd.read_csv('../data/customers.csv')
transactions_df = pd.read_csv('../data/transactions.csv')
products_df = pd.read_csv('../data/products_held.csv')
catalog_df = pd.read_csv('../data/product_catalog.csv')

print("✅ Data loaded!")
print(f"Customers: {len(customers_df):,}")
print(f"Transactions: {len(transactions_df):,}")

✅ Data loaded!
Customers: 10,000
Transactions: 300,123


In [2]:
# Calculate spending % per category per customer
spending = transactions_df.groupby(['customer_id', 'category'])['amount'].sum().reset_index()
spending_pivot = spending.pivot(index='customer_id', columns='category', values='amount').fillna(0)

# Convert to percentages
spending_pivot = spending_pivot.div(spending_pivot.sum(axis=1), axis=0)
spending_pivot.columns = [f'{col}_pct' for col in spending_pivot.columns]
spending_pivot = spending_pivot.reset_index()

# Merge with customer demographics
customer_features = customers_df[['customer_id', 'age', 'income', 'credit_score', 'segment']].merge(
    spending_pivot, on='customer_id', how='left'
).fillna(0)

print("✅ Customer features built!")
print(f"Shape: {customer_features.shape}")
print(customer_features.head(3))

✅ Customer features built!
Shape: (10000, 13)
  customer_id  age  income  credit_score             segment  dining_pct  \
0     C_00001   22   58024           651  young_professional    0.061605   
1     C_00002   32   75741           631  young_professional    0.040127   
2     C_00003   23   54328           649  young_professional    0.025835   

   entertainment_pct  groceries_pct  healthcare_pct  rent_pct  shopping_pct  \
0           0.035626            0.0        0.026736  0.222221      0.037366   
1           0.038619            0.0        0.023842  0.111980      0.081849   
2           0.023324            0.0        0.003107  0.120023      0.028758   

   travel_pct  utilities_pct  
0    0.595643       0.020803  
1    0.671467       0.032115  
2    0.788138       0.010815  


In [3]:
# Define what each product's ideal customer looks like
# Based on spending patterns and demographics

product_profiles = {
    'P001': {'name': 'checking_account',    'travel': 0.1, 'dining': 0.1, 'groceries': 0.2, 'rent': 0.3, 'shopping': 0.1, 'entertainment': 0.1, 'healthcare': 0.05, 'utilities': 0.05, 'min_income': 0,      'min_credit': 580},
    'P002': {'name': 'savings_account',     'travel': 0.1, 'dining': 0.1, 'groceries': 0.2, 'rent': 0.2, 'shopping': 0.1, 'entertainment': 0.1, 'healthcare': 0.1,  'utilities': 0.1,  'min_income': 20000,  'min_credit': 580},
    'P003': {'name': 'travel_credit_card',  'travel': 0.4, 'dining': 0.2, 'groceries': 0.1, 'rent': 0.1, 'shopping': 0.1, 'entertainment': 0.05,'healthcare': 0.02, 'utilities': 0.03, 'min_income': 60000,  'min_credit': 700},
    'P004': {'name': 'cashback_credit_card','travel': 0.1, 'dining': 0.2, 'groceries': 0.3, 'rent': 0.1, 'shopping': 0.2, 'entertainment': 0.05,'healthcare': 0.02, 'utilities': 0.03, 'min_income': 30000,  'min_credit': 640},
    'P005': {'name': 'personal_loan',       'travel': 0.1, 'dining': 0.1, 'groceries': 0.2, 'rent': 0.3, 'shopping': 0.1, 'entertainment': 0.05,'healthcare': 0.05, 'utilities': 0.1,  'min_income': 40000,  'min_credit': 640},
    'P006': {'name': 'home_loan',           'travel': 0.05,'dining': 0.1, 'groceries': 0.2, 'rent': 0.4, 'shopping': 0.1, 'entertainment': 0.05,'healthcare': 0.05, 'utilities': 0.05, 'min_income': 60000,  'min_credit': 680},
    'P007': {'name': 'investment_fund',     'travel': 0.2, 'dining': 0.1, 'groceries': 0.1, 'rent': 0.1, 'shopping': 0.1, 'entertainment': 0.1, 'healthcare': 0.1,  'utilities': 0.1,  'min_income': 80000,  'min_credit': 680},
    'P008': {'name': 'fixed_deposit',       'travel': 0.1, 'dining': 0.1, 'groceries': 0.2, 'rent': 0.1, 'shopping': 0.1, 'entertainment': 0.1, 'healthcare': 0.1,  'utilities': 0.2,  'min_income': 30000,  'min_credit': 620},
}

print("✅ Product profiles defined!")
for pid, profile in product_profiles.items():
    print(f"{pid}: {profile['name']}")

✅ Product profiles defined!
P001: checking_account
P002: savings_account
P003: travel_credit_card
P004: cashback_credit_card
P005: personal_loan
P006: home_loan
P007: investment_fund
P008: fixed_deposit


In [4]:
# spending categories
categories = ['travel', 'dining', 'groceries', 'rent', 'shopping', 'entertainment', 'healthcare', 'utilities']

# Build product vector matrix
product_vectors = []
product_ids = []
for pid, profile in product_profiles.items():
    vector = [profile[cat] for cat in categories]
    product_vectors.append(vector)
    product_ids.append(pid)

product_matrix = np.array(product_vectors)

# Build customer spending vector matrix
feature_cols = [f'{cat}_pct' for cat in categories]
customer_matrix = customer_features[feature_cols].values

# Calculate cosine similarity
similarity_matrix = cosine_similarity(customer_matrix, product_matrix)

# Convert to DataFrame
similarity_df = pd.DataFrame(
    similarity_matrix,
    columns=product_ids,
    index=customer_features['customer_id']
)

print("✅ Cosine similarity calculated!")
print(f"Shape: {similarity_df.shape}")
print("\nSample similarities for C_00001:")
print(similarity_df.loc['C_00001'].sort_values(ascending=False))

✅ Cosine similarity calculated!
Shape: (10000, 8)

Sample similarities for C_00001:
P003    0.900722
P007    0.749733
P001    0.529227
P005    0.526466
P002    0.509041
P006    0.431279
P008    0.425153
P004    0.369992
Name: C_00001, dtype: float64


In [5]:
# Get products each customer already owns
owned_products = products_df.groupby('customer_id')['product_id'].apply(set).to_dict()

# Generate top 3 recommendations per customer
# excluding products they already own
cb_results = []

for customer_id in customer_features['customer_id']:
    owned = owned_products.get(customer_id, set())
    scores = similarity_df.loc[customer_id]
    
    # Filter out already owned products
    unowned_scores = scores[~scores.index.isin(owned)]
    
    # Get top 3
    top3 = unowned_scores.sort_values(ascending=False).head(3)
    
    for product_id, score in top3.items():
        product_name = product_profiles[product_id]['name']
        cb_results.append({
            'customer_id': customer_id,
            'product_id': product_id,
            'product_name': product_name,
            'cb_score': round(score, 4)
        })

cb_df = pd.DataFrame(cb_results)

print("✅ Content-based recommendations generated!")
print(f"Total: {len(cb_df):,}")
print("\nSample:")
print(cb_df.head(12))

✅ Content-based recommendations generated!
Total: 29,966

Sample:
   customer_id product_id        product_name  cb_score
0      C_00001       P003  travel_credit_card    0.9007
1      C_00001       P007     investment_fund    0.7497
2      C_00001       P005       personal_loan    0.5265
3      C_00002       P007     investment_fund    0.7314
4      C_00002       P002     savings_account    0.4313
5      C_00002       P005       personal_loan    0.4138
6      C_00003       P003  travel_credit_card    0.8724
7      C_00003       P007     investment_fund    0.6751
8      C_00003       P002     savings_account    0.3748
9      C_00004       P005       personal_loan    0.8483
10     C_00004       P006           home_loan    0.8233
11     C_00004       P002     savings_account    0.8219


In [6]:
# Save content-based recommendations
cb_df.to_csv('../data/cb_recommendations.csv', index=False)

# Compare ALS vs Content-Based for same customer
print("=" * 50)
print("COMPARISON: C_00001")
print("=" * 50)

print("\n🤖 ALS Recommendations:")
als_df = pd.read_csv('../data/als_recommendations.csv')
print(als_df[als_df['customer_id'] == 'C_00001'][['product_name', 'score']])

print("\n📊 Content-Based Recommendations:")
print(cb_df[cb_df['customer_id'] == 'C_00001'][['product_name', 'cb_score']])

print("\n✅ Both models agree on travel_credit_card!")
print("✅ Content-based saved!")

COMPARISON: C_00001

🤖 ALS Recommendations:
       product_name   score
0   savings_account  0.9451
1  checking_account  0.9353
2     fixed_deposit  0.0391

📊 Content-Based Recommendations:
         product_name  cb_score
0  travel_credit_card    0.9007
1     investment_fund    0.7497
2       personal_loan    0.5265

✅ Both models agree on travel_credit_card!
✅ Content-based saved!
